Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/lucianommartins/lab-sabadao/blob/main/examples/notebooks/04_academic_evaluations_and_reasoning.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench academic evaluations, reasoning mode, and multimodal budgets

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This notebook explores how to measure the reasoning accuracy of foundation models using `gbench --evals`. You will run native Python evaluation harnesses for science, mathematics, function calling, and multimodal understanding against a local Ollama server, configuring Chain-of-Thought reasoning (`--eval-thinking`), few-shot prompts (`--eval-n-shot`), and vision soft-token compression (`--eval-max-soft-tokens`).

## Learning objectives

1. Configure Ollama to serve a quantized Google Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 8192 tokens.
2. Execute native Python academic evaluation suites (`gpqa`, `mmlu`, `gsm8k`, `bfcl`, `mmmu_pro`).
3. Enable Chain-of-Thought reasoning mode (`--eval-thinking`) and analyze impact on reasoning accuracy.
4. Test vision soft-token budgets (`--eval-max-soft-tokens 1120`) for multimodal evaluations.
5. Evaluate few-shot prompt accuracy (`--eval-n-shot 5` vs 0-shot).
6. Perform a clean session shutdown to terminate background servers and reclaim hardware memory.

## Useful resources

* [lab-sabadao GitHub repository](https://github.com/lucianommartins/lab-sabadao)
* [GPQA science evaluation benchmark](https://github.com/idavidrein/gpqa)
* [GSM8K math reasoning benchmark](https://github.com/openai/grade-school-math)
* [BFCL Berkeley function calling leaderboard](https://gorilla.cs.berkeley.edu/leaderboard.html)

## 1. Environment setup and installation

We clone the `lab-sabadao` repository from GitHub, change directory into the project root (`%cd lab-sabadao`), and install the package in editable mode (`%pip install -e .`). This builds and links the `gbench` CLI executable without installing unnecessary development linters.

In [ ]:
import os, sys
from pathlib import Path

# Safe environment setup: Always normalize to top-level repository
if Path("/content").exists():
    %cd -q /content
    if not Path("/content/lab-sabadao").is_dir():
        !git clone https://github.com/lucianommartins/lab-sabadao.git
    %cd -q /content/lab-sabadao
else:
    if not Path("pyproject.toml").is_file() and not Path("gbench").is_dir():
        if not Path("lab-sabadao").is_dir():
            !git clone https://github.com/lucianommartins/lab-sabadao.git
        %cd lab-sabadao

%pip install -e . -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available evaluation benchmark suites and pillars
!gbench --list evals
!gbench --list pillars

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil

if not shutil.which("ollama"):
    print("Installing Ollama locally...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama binary already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Writing custom Modelfile for QAT model

We create an Ollama `Modelfile.qat` that configures our quantized Google Gemma 4 model (`hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:latest`) with explicit parameters:
* **`num_ctx 8192`**: Context window of 8192 tokens.
* **`SYSTEM prompt`**: System instruction defining Gemma 4 AI assistant capabilities.

In [ ]:
HF_MODEL_ID = "hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:UD-Q4_K_XL"
modelfile_content = f"""FROM {HF_MODEL_ID}
PARAMETER num_ctx 8192
SYSTEM "You are a helpful Gemma 4 AI assistant with reasoning, vision, and tool calling capabilities."
"""
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write(modelfile_content)
print("Created Modelfile.qat with valid Ollama parameters (num_ctx 8192, SYSTEM prompt).")

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) using `ollama create -f Modelfile.qat`. This pulls the GGUF weights from Hugging Face Hub if not already cached. We then run a quick generation test (`ollama run`) to verify that the model loads into hardware memory and generates tokens correctly.

In [ ]:
import requests

MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from Modelfile.qat...")
!ollama create {MODEL_TAG} -f Modelfile.qat

print("Running quick generation smoke test via Ollama API (cold load into GPU VRAM)...")
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_TAG, "prompt": "Reply with the single word: READY.", "stream": False},
    timeout=300,
)
print("Smoke test response:", resp.json().get("response", "").strip())

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in resp.json().get("data", [])]
print("Available REST models:", models)

## 7. Running academic benchmarks with Chain-of-Thought reasoning

We can inspect all registered evaluation suites and capability pillars via `!gbench --list evals` and `!gbench --list pillars`.

Then we execute `gbench --evals-only` with `--eval-thinking` enabled. This evaluates our target QAT model (`gemma4-qat:4b`) across open academic reasoning suites (`gsm8k`, `mmlu`), using `--eval-limit 5` for quick turnaround while instructing the model to generate explicit `<think>...</think>` reasoning chains before producing final structured answers.

In [ ]:
# List all registered evaluation benchmark suites
!gbench --list evals

# Execute academic evaluations with thinking enabled and limited sample size
!gbench --evals-only \
        --models gemma4-qat:4b \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --evals gsm8k mmlu \
        --eval-limit 5 \
        --eval-thinking \
        --results-dir ./results_evals_cot

## 8. Budgeting vision soft tokens for multimodal benchmarks

When benchmarking vision suites like `mmmu_pro` or `math_vista`, Google Gemma 4 requires sufficient soft-token budgeting (`1120` tokens) to avoid downsampling complex charts or geometry figures.

In [ ]:
!gbench --evals-only \
        --models gemma4-qat:4b \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --evals mmmu_pro \
        --eval-limit 5 \
        --eval-max-soft-tokens 1120 \
        --results-dir ./results_evals_mm

## 9. Analyzing evaluation scores and accuracies

We load `eval_results.json` into Python to inspect per-task accuracies across academic evaluation suites.

In [ ]:
import json, glob, os
from pathlib import Path
import pandas as pd

def load_eval_results(results_base_dir):
    base = Path(results_base_dir)
    run_dirs = sorted([d for d in base.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True) if base.exists() else []
    if not run_dirs:
        return pd.DataFrame()
    latest_dir = run_dirs[0]
    summary_file = latest_dir / "summary.json"
    records = []
    if summary_file.exists():
        with open(summary_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        for m in data.get("models", []):
            if "eval_results" in m:
                for suite, metrics in m["eval_results"].items():
                    records.append({
                        "model": m.get("model", "gemma4-qat:4b"),
                        "suite": suite,
                        "score": metrics.get("score", metrics.get("accuracy", "N/A")),
                        "details": str(metrics)[:60]
                    })
    if not records:
        eval_files = sorted(latest_dir.glob("eval_*.json")) or sorted(latest_dir.glob("*.json"))
        for ef in eval_files:
            with open(ef, "r", encoding="utf-8") as f:
                data = json.load(f)
            results = data.get("results", {}) if isinstance(data, dict) else {}
            for suite, metrics in results.items():
                records.append({
                    "suite": suite,
                    "score": metrics.get("score", metrics.get("accuracy", "N/A")),
                    "details": str(metrics)[:60]
                })
    return pd.DataFrame(records)

print("=== Reasoning Benchmark Results (CoT) ===")
df_evals = load_eval_results("./results_evals_cot")
if not df_evals.empty:
    display(df_evals)
else:
    print("No reasoning eval results found.")

print("\n=== Multimodal Benchmark Results ===")
df_evals_mm = load_eval_results("./results_evals_mm")
if not df_evals_mm.empty:
    display(df_evals_mm)
else:
    print("No multimodal eval results found.")

## 10. Session cleanup and server shutdown

We terminate background Ollama server processes and remove temporary Modelfiles.

In [ ]:
import subprocess, os

subprocess.run(["pkill", "-f", "ollama"], check=False)
if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")
print("Session cleanup complete.")